In [3]:
import pandas as pd
import numpy as np
import re

In [137]:
df = pd.read_csv("output/attendance.csv")
df.head()

,attendance_id,employee_id,attendance_date,status,check_in,check_out,overtime_hours
0,ATT000001,EMP01957,2025-06-21,Absent,NaN,NaN,0.0
1,ATT000002,EMP02276,2025-04-08,Present,09:40,17:03,0.9
2,ATT000003,EMP02293,2025-12-11,Present,10:13,18:47,0.0
3,ATT000004,EMP02386,2025-04-23,Present,08:43,19:52,0.0
4,ATT000005,EMP01076,2026-07-21,Present,09:48,20:59,0.0


In [90]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Rows: 42630
Columns: 7

Column Names:
['attendance_id', 'employee_id', 'attendance_date', 'status', 'check_in', 'check_out', 'overtime_hours']

Data Types:
attendance_id          str
employee_id            str
attendance_date        str
status                 str
check_in               str
check_out              str
overtime_hours     float64
dtype: object


In [138]:
print("Duplicate rows:", 
df.duplicated().sum())

Duplicate rows: 630


In [92]:
df = df.drop_duplicates()
print("Row after removing duplicates:", len(df))

Row after removing duplicates: 42000


In [139]:
print(df.columns.tolist())
print(df.dtypes)
display(df.head(10))

['attendance_id', 'employee_id', 'attendance_date', 'status', 'check_in', 'check_out', 'overtime_hours']
attendance_id          str
employee_id            str
attendance_date        str
status                 str
check_in               str
check_out              str
overtime_hours     float64
dtype: object


,attendance_id,employee_id,attendance_date,status,check_in,check_out,overtime_hours
0,ATT000001,EMP01957,2025-06-21,Absent,NaN,NaN,0.0
1,ATT000002,EMP02276,2025-04-08,Present,09:40,17:03,0.9
2,ATT000003,EMP02293,2025-12-11,Present,10:13,18:47,0.0
3,ATT000004,EMP02386,2025-04-23,Present,08:43,19:52,0.0
4,ATT000005,EMP01076,2026-07-21,Present,09:48,20:59,0.0
5,ATT000006,EMP02331,2025-06-13,Absent,NaN,NaN,0.0
6,ATT000007,EMP02033,2025-03-01,On Leave,NaN,NaN,0.0
7,ATT000008,EMP01211,2025-05-31,Present,09:51,19:22,0.0
8,ATT000009,EMP00760,2025-09-07,Present,08:42,17:45,0.0
9,ATT000010,EMP03088,2025-03-05,Present,10:02,20:54,0.0


In [140]:
df["attendance_date"] = pd.to_datetime(
    df["attendance_date"],
    errors="coerce"
)


In [95]:
print(df.dtypes)

attendance_id                 str
employee_id                   str
attendance_date    datetime64[us]
status                        str
check_in                      str
check_out                     str
overtime_hours            float64
dtype: object


In [141]:
df["check_in"] = pd.to_datetime(
    df["check_in"].astype("string").str.strip(),
    errors="coerce"
).dt.time

df["check_out"] = pd.to_datetime(
    df["check_out"].astype("string").str.strip(),
    errors="coerce"
).dt.time

C:\Users\kiran\AppData\Local\Temp\ipykernel_22504\1322964717.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["check_in"] = pd.to_datetime(
C:\Users\kiran\AppData\Local\Temp\ipykernel_22504\1322964717.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["check_out"] = pd.to_datetime(


In [142]:
print(df.dtypes)

attendance_id                 str
employee_id                   str
attendance_date    datetime64[us]
status                        str
check_in                   object
check_out                  object
overtime_hours            float64
dtype: object


In [143]:
print(df["status"].value_counts(dropna=False))

status
Present           30691
Work From Home     4293
On Leave           3850
Absent             2542
Half Day           1254
Name: count, dtype: int64


In [144]:
valid_status = [
    "Present",
    "Absent",
    "On Leave",
    "Work From Home",
    "Half Day"
]

invalid_status = df[
    ~df["status"].isin(valid_status)
]

print("Invalid status records:", len(invalid_status))

Invalid status records: 0


In [145]:
invalid_attendance_id = df[
    ~df["attendance_id"].str.match(r"^ATT\d{6}$", na=False)
]

print("Invalid Attendance IDs:", len(invalid_attendance_id))

Invalid Attendance IDs: 0


In [146]:
invalid_employee_id = df[
    ~df["employee_id"].str.match(r"^EMP\d{5}$", na=False)
]

print("Invalid Employee IDs:", len(invalid_employee_id))

Invalid Employee IDs: 0


In [147]:
df["attendance_date"] = pd.to_datetime(
    df["attendance_date"],
    errors="coerce"
)

print("Invalid dates:", df["attendance_date"].isna().sum())

Invalid dates: 0


In [148]:
missing = df.isnull().sum()
print(missing)

attendance_id         0
employee_id           0
attendance_date       0
status                0
check_in           6392
check_out          6392
overtime_hours        0
dtype: int64


In [149]:
present_missing = df[
    (df["status"] == "Present") &
    (df["check_in"].isna() | df["check_out"].isna())
]

print("Present records with missing time:", len(present_missing))

Present records with missing time: 0


In [150]:
wfh_missing = df[
    (df["status"] == "Work From Home") &
    (df["check_in"].isna() | df["check_out"].isna())
]

print("WFH records with missing time:", len(wfh_missing))

WFH records with missing time: 0


In [151]:
halfday_missing = df[
    (df["status"] == "Half Day") &
    (df["check_in"].isna() | df["check_out"].isna())
]

print("Half Day records with missing time:", len(halfday_missing))

Half Day records with missing time: 0


In [152]:
print(df["overtime_hours"].describe())

count    42630.000000
mean         0.435433
std          1.408897
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         15.000000
Name: overtime_hours, dtype: float64


In [153]:
print(
    "Negative overtime:",
    (df["overtime_hours"]<0).sum()
)

Negative overtime: 0


In [156]:
invalid_overtime = df[
    df["status"].isin(["Absent", "On Leave"]) &
    (df["overtime_hours"] > 0)
]

print("Invalid overtime records:", len(invalid_overtime))

Invalid overtime records: 68


In [155]:
df.loc[
    (df["overtime_hours"] < 0) |
    (df["overtime_hours"] > 24),
    "overtime_hours"
] = np.nan

In [157]:
print(
    "Invalid overtime after cleaning:",
    (
        (df["overtime_hours"] < 0) |
        (df["overtime_hours"] > 24)
    ).sum()
)

Invalid overtime after cleaning: 0


In [158]:
df.loc[
    df["status"].isin(["Absent", "On Leave"]),
    "overtime_hours"
] = 0

In [159]:
df.loc[
    df["overtime_hours"] < 0,
    "overtime_hours"
] = 0

In [160]:
invalid_overtime = df[
    df["status"].isin(["Absent", "On Leave"]) &
    (df["overtime_hours"] > 0)
]

print("Invalid overtime records:", len(invalid_overtime))

Invalid overtime records: 0


In [162]:
duplicate_employee_date = df[
    df.duplicated(
        subset=["employee_id", "attendance_date"],
        keep=False
    )
]

print(
    "Duplicate employee-date records:",
    len(duplicate_employee_date)
)

Duplicate employee-date records: 2179


In [163]:
duplicate_attendance_ids = df[
    df["attendance_id"].duplicated(keep=False)
]

print(
    "Duplicate attendance IDs:",
    duplicate_attendance_ids["attendance_id"].nunique()
)

Duplicate attendance IDs: 624


In [164]:
df["year"] = df["attendance_date"].dt.year
df["month"] = df["attendance_date"].dt.month
df["month_name"] = df["attendance_date"].dt.month_name()
df["day_name"] = df["attendance_date"].dt.day_name()

In [165]:
print("Final Missing Values:")
print(df.isnull().sum())

Final Missing Values:
attendance_id         0
employee_id           0
attendance_date       0
status                0
check_in           6392
check_out          6392
overtime_hours        0
year                  0
month                 0
month_name            0
day_name              0
dtype: int64


In [166]:
print("Final duplicate rows:",df.duplicated().sum())

Final duplicate rows: 630


In [167]:
print(df.dtypes)

attendance_id                 str
employee_id                   str
attendance_date    datetime64[us]
status                        str
check_in                   object
check_out                  object
overtime_hours            float64
year                        int32
month                       int32
month_name                    str
day_name                      str
dtype: object


In [168]:
print("Final Shape:",df.shape)
display(df.head(10))

Final Shape: (42630, 11)


,attendance_id,employee_id,attendance_date,status,check_in,check_out,overtime_hours,year,month,month_name,day_name
0,ATT000001,EMP01957,2025-06-21,Absent,NaT,NaT,0.0,2025,6,June,Saturday
1,ATT000002,EMP02276,2025-04-08,Present,09:40:00,17:03:00,0.9,2025,4,April,Tuesday
2,ATT000003,EMP02293,2025-12-11,Present,10:13:00,18:47:00,0.0,2025,12,December,Thursday
3,ATT000004,EMP02386,2025-04-23,Present,08:43:00,19:52:00,0.0,2025,4,April,Wednesday
4,ATT000005,EMP01076,2026-07-21,Present,09:48:00,20:59:00,0.0,2026,7,July,Tuesday
5,ATT000006,EMP02331,2025-06-13,Absent,NaT,NaT,0.0,2025,6,June,Friday
6,ATT000007,EMP02033,2025-03-01,On Leave,NaT,NaT,0.0,2025,3,March,Saturday
7,ATT000008,EMP01211,2025-05-31,Present,09:51:00,19:22:00,0.0,2025,5,May,Saturday
8,ATT000009,EMP00760,2025-09-07,Present,08:42:00,17:45:00,0.0,2025,9,September,Sunday
9,ATT000010,EMP03088,2025-03-05,Present,10:02:00,20:54:00,0.0,2025,3,March,Wednesday


In [169]:
# Exact duplicate rows remove karo

df = df.drop_duplicates()

print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Rows after cleaning:", len(df))

Duplicate rows after cleaning: 0
Rows after cleaning: 42000


In [170]:
print("Exact duplicate rows:", df.duplicated().sum())

print("Duplicate Attendance IDs:",
      df["attendance_id"].duplicated().sum())

print("Total rows:", len(df))

Exact duplicate rows: 0
Duplicate Attendance IDs: 0
Total rows: 42000


In [171]:
print(df.isnull().sum())

attendance_id         0
employee_id           0
attendance_date       0
status                0
check_in           6293
check_out          6293
overtime_hours        0
year                  0
month                 0
month_name            0
day_name              0
dtype: int64


In [172]:
print("Exact duplicate rows:",
      df.duplicated().sum())

Exact duplicate rows: 0


In [173]:
print("Duplicate Attendance IDs:",
      df["attendance_id"].duplicated().sum())

Duplicate Attendance IDs: 0


In [174]:
Q1 = df["overtime_hours"].quantile(0.25)
Q3 = df["overtime_hours"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["overtime_hours"] < lower) |
    (df["overtime_hours"] > upper)
]

print("Overtime outliers:", len(outliers))
print("Lower limit:", lower)
print("Upper limit:", upper)

Overtime outliers: 6428
Lower limit: 0.0
Upper limit: 0.0


In [175]:
print(df.dtypes)

attendance_id                 str
employee_id                   str
attendance_date    datetime64[us]
status                        str
check_in                   object
check_out                  object
overtime_hours            float64
year                        int32
month                       int32
month_name                    str
day_name                      str
dtype: object


In [178]:
df = df.sort_values(
    by=["employee_id", "attendance_date"]
)

df = df.drop_duplicates(
    subset=["employee_id", "attendance_date"],
    keep="first"
)

In [179]:
df.duplicated(
    subset=["employee_id", "attendance_date"]
).sum()

np.int64(0)

In [180]:
df.shape

(41529, 11)

In [206]:
df.describe()

,age,experience_years,monthly_basic_salary,satisfaction_score
count,3264.000000,3264.000000,3264.000000,3125.000000
mean,40.602635,15.335631,128200.061275,2.976640
std,12.197647,9.053449,92210.153120,1.400029
min,16.000000,0.000000,25000.000000,1.000000
25%,30.000000,7.700000,45000.000000,2.000000
50%,41.000000,15.200000,100000.000000,3.000000
75%,50.000000,23.000000,200000.000000,4.000000
max,99.000000,49.600000,320000.000000,5.000000


In [181]:
df.to_csv(
    "attendance_cleaned.csv",
    index=False
)

In [182]:
df = pd.read_csv("output/departments.csv")
df.head()

,department_id,department_name,description,budget_inr
0,DEP001,Engineering,Software & Product Engineering,5000000
1,DEP002,Sales,Sales & Business Development,3500000
2,DEP003,Marketing,Marketing & Brand,7500000
3,DEP004,Human Resources,People Operations,15000000
4,DEP005,Finance,Finance & Accounts,2000000


In [183]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   department_id    12 non-null     str  
 1   department_name  12 non-null     str  
 2   description      12 non-null     str  
 3   budget_inr       12 non-null     int64
dtypes: int64(1), str(3)
memory usage: 516.0 bytes


In [184]:
df.isnull().sum()

department_id      0
department_name    0
description        0
budget_inr         0
dtype: int64

In [185]:
df.isna().all(axis=1).sum()

np.int64(0)

In [186]:
df.columns

Index(['department_id', 'department_name', 'description', 'budget_inr'], dtype='str')

In [191]:
df.duplicated().sum()

np.int64(0)

In [192]:
df["department_id"].duplicated().sum()

np.int64(0)

In [193]:
df["department_name"].duplicated().sum()

np.int64(0)

In [194]:
df["budget_inr"].dtype

dtype('int64')

In [195]:
df["budget_inr"] = pd.to_numeric(df["budget_inr"], errors="coerce")

df["budget_inr"].isnull().sum()

np.int64(0)

In [196]:
(df["budget_inr"]<0).sum()

np.int64(0)

In [197]:
df.head(12)

,department_id,department_name,description,budget_inr
0,DEP001,Engineering,Software & Product Engineering,5000000
1,DEP002,Sales,Sales & Business Development,3500000
2,DEP003,Marketing,Marketing & Brand,7500000
3,DEP004,Human Resources,People Operations,15000000
4,DEP005,Finance,Finance & Accounts,2000000
5,DEP006,Customer Support,Customer Success,2000000
6,DEP007,Operations,Operations & Logistics,10000000
7,DEP008,IT Infrastructure,IT & Systems,2000000
8,DEP009,Legal,Legal & Compliance,5000000
9,DEP010,Procurement,Procurement & Vendor Management,10000000


In [205]:
df.describe()

,age,experience_years,monthly_basic_salary,satisfaction_score
count,3264.000000,3264.000000,3264.000000,3125.000000
mean,40.602635,15.335631,128200.061275,2.976640
std,12.197647,9.053449,92210.153120,1.400029
min,16.000000,0.000000,25000.000000,1.000000
25%,30.000000,7.700000,45000.000000,2.000000
50%,41.000000,15.200000,100000.000000,3.000000
75%,50.000000,23.000000,200000.000000,4.000000
max,99.000000,49.600000,320000.000000,5.000000


In [198]:
df.to_csv(
    "departments_cleaned.csv",
    index=False
)

In [3]:
df = pd.read_csv("output/employees.csv")
df.head()

,employee_id,employee_name,gender,dob,age,marital_status,city,state,phone,email,department_id,designation,education,experience_years,date_of_joining,employment_status,exit_date,monthly_basic_salary,satisfaction_score
0,EMP00001,Vivaan Gupta,Male,1979-02-08,47,Single,Indore,Madhya Pradesh,+916539082113,vivaan.gupta106@yahoo.com,DEP010,Deputy General Manager,B.Sc,1.8,2017-01-13,Resigned,2022-01-15,250000,3.0
1,EMP00002,Suresh Ghosh,Other,1985-08-19,41,Single,Indore,Madhya Pradesh,+918587291471,suresh.ghosh986@gmail.com,DEP011,Senior Associate,Diploma,9.0,2025-03-01,Active,NaN,45000,5.0
2,EMP00003,Rekha Malhotra,Female,2001-02-09,25,Divorced,Chandigarh,Chandigarh,+918783936422,rekha.malhotra401@hotmail.com,DEP008,Senior Associate,B.A.,1.8,2025-08-19,Terminated,NaN,80000,4.0
3,EMP00004,Nisha Iyer,Female,1983-11-13,43,Single,Indore,Madhya Pradesh,+91-121563,nisha.iyer974@rediffmail.com,DEP010,General Manager,B.Tech,2.5,2020-03-05,Active,NaN,45000,4.0
4,EMP00005,Sneha Yadav,Other,1999-08-21,27,Single,Nagpur,Maharashtra,+916155960236,sneha.yadav373@rediffmail.com,DEP008,Senior Associate,B.Com,5.7,2023-12-25,Active,NaN,60000,4.0


In [49]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

Rows: 3200
Columns: 19

Column Names:
['employee_id', 'employee_name', 'gender', 'dob', 'age', 'marital_status', 'city', 'state', 'phone', 'email', 'department_id', 'designation', 'education', 'experience_years', 'date_of_joining', 'employment_status', 'exit_date', 'monthly_basic_salary', 'satisfaction_score']


In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           3200 non-null   str    
 1   employee_name         3200 non-null   str    
 2   gender                3200 non-null   str    
 3   dob                   3200 non-null   str    
 4   age                   3200 non-null   int64  
 5   marital_status        3200 non-null   str    
 6   city                  3200 non-null   str    
 7   state                 3200 non-null   str    
 8   phone                 2958 non-null   string 
 9   email                 3110 non-null   string 
 10  department_id         3200 non-null   str    
 11  designation           3200 non-null   str    
 12  education             3200 non-null   str    
 13  experience_years      3065 non-null   float64
 14  date_of_joining       3200 non-null   str    
 15  employment_status     3200 non-n

In [51]:
date_columns = ["dob", "date_of_joining", "exit_date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   employee_id           3200 non-null   str           
 1   employee_name         3200 non-null   str           
 2   gender                3200 non-null   str           
 3   dob                   3200 non-null   datetime64[us]
 4   age                   3200 non-null   int64         
 5   marital_status        3200 non-null   str           
 6   city                  3200 non-null   str           
 7   state                 3200 non-null   str           
 8   phone                 2958 non-null   string        
 9   email                 3110 non-null   string        
 10  department_id         3200 non-null   str           
 11  designation           3200 non-null   str           
 12  education             3200 non-null   str           
 13  experience_years      3065 no

In [52]:
df.isnull().sum()

employee_id                0
employee_name              0
gender                     0
dob                        0
age                        0
marital_status             0
city                       0
state                      0
phone                    242
email                     90
department_id              0
designation                0
education                  0
experience_years         135
date_of_joining            0
employment_status          0
exit_date               2532
monthly_basic_salary       0
satisfaction_score         0
dtype: int64

In [53]:
print("Duplicate rows:",
     df.duplicated().sum())

Duplicate rows: 0


In [54]:
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 3200


In [55]:
text_columns = df.select_dtypes(include=["object"]).columns

for col in text_columns:
    df[col] = df[col].str.strip()

C:\Users\kiran\AppData\Local\Temp\ipykernel_24220\1908794907.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include=["object"]).columns


In [56]:
date_columns = [
    "dob",
    "date_of_joining",
    "exit_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [57]:
df[date_columns].dtypes

dob                datetime64[us]
date_of_joining    datetime64[us]
exit_date          datetime64[us]
dtype: object

In [58]:
numeric_columns = [
    "age",
    "experience_years",
    "monthly_basic_salary",
    "satisfaction_score"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [59]:
today = pd.Timestamp("2026-09-09")

dob = df["dob"]

df["age"] = (
    today.year - dob.dt.year
    - (
        (today.month < dob.dt.month)
        |
        (
            (today.month == dob.dt.month)
            & (today.day < dob.dt.day)
        )
    ).astype(int)
)

In [60]:
df[["dob", "age"]].head(10)

,dob,age
0,1979-02-08,47
1,1985-08-19,41
2,2001-02-09,25
3,1983-11-13,42
4,1999-08-21,27
5,1989-08-27,37
6,1983-12-01,42
7,1984-02-26,42
8,1992-05-17,34
9,1996-03-05,30


In [61]:
known_domains = [
    "gmail.com",
    "yahoo.com",
    "hotmail.com",
    "outlook.com",
    "rediffmail.com"
]

def clean_email(email):
    
    if pd.isna(email):
        return np.nan
    
    email = str(email).strip().lower()
    
    # Double @ ko single @
    email = email.replace("@@", "@")
    
    # Missing @ ko repair karna
    if "@" not in email:
        for domain in known_domains:
            if domain in email:
                email = email.replace(domain, "@" + domain)
                break
    
    # Missing .com ko repair karna
    for domain in ["gmail", "yahoo", "hotmail", "outlook", "rediffmail"]:
        if re.search(r"@" + re.escape(domain) + r"$", email):
            email = email + ".com"
            break
    
    # Final validation
    if re.fullmatch(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        email
    ):
        return email
    
    return np.nan


df["email"] = df["email"].apply(clean_email)

In [62]:
df['email'] = (
    df['email']
    .astype("string")
    .str.strip()
    .str.lower()
)

print("Missing emails:", df['email'].isna().sum())
print("Invalid emails:", 
      (~df['email'].isna() & 
       ~df['email'].str.match(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')).sum())

Missing emails: 90
Invalid emails: 0


In [37]:
duplicate_email_mask = (
    df['email'].notna() &
    df['email'].duplicated(keep='first')
)

print("Duplicate email records:", duplicate_email_mask.sum())

df.loc[duplicate_email_mask, 'email'] = pd.NA

print("Duplicate emails after cleaning:",
      df['email'].dropna().duplicated().sum())

Duplicate email records: 4
Duplicate emails after cleaning: 0


In [38]:
df["email"].head(20)

0            vivaan.gupta106@yahoo.com
1            suresh.ghosh986@gmail.com
2        rekha.malhotra401@hotmail.com
3         nisha.iyer974@rediffmail.com
4        sneha.yadav373@rediffmail.com
5        arjun.singh338@rediffmail.com
6     krishna.kapoor861@rediffmail.com
7            suresh.patel819@gmail.com
8     krishna.sharma835@rediffmail.com
9        myra.chopra621@rediffmail.com
10          gaurav.kapoor765@yahoo.com
11            divya.yadav277@yahoo.com
12        sarika.rao727@rediffmail.com
13           divya.shetty527@yahoo.com
14             divya.menon57@yahoo.com
15            ananya.joshi@outlook.com
16           vikram.kapoor88@yahoo.com
17            anil.sharma491@gmail.com
18                                <NA>
19         arjun.pillai123@hotmail.com
Name: email, dtype: string

In [69]:
def clean_phone(phone):
    
    if pd.isna(phone):
        return np.nan
    
    digits = re.sub(r"\D", "", str(phone))
    
    # Already +91 + 10 digit mobile number
    if (
        len(digits) == 12
        and digits.startswith("91")
        and digits[2] in "6789"
    ):
        return "+91" + digits[2:]
    
    # 10 digit Indian mobile number
    if len(digits) == 10 and digits[0] in "6789":
        return "+91" + digits
    
    return np.nan


df["phone"] = df["phone"].apply(clean_phone)

In [71]:
# Phone number cleaning

df["phone"] = df["phone"].astype("string").str.strip()

# Remove +91, spaces, -, brackets etc.
df["phone"] = df["phone"].str.replace(r"\D", "", regex=True)

# If number has 91 + 10 digits, keep only 10 digits
df.loc[
    df["phone"].str.len().eq(12) &
    df["phone"].str.startswith("91", na=False),
    "phone"
] = df.loc[
    df["phone"].str.len().eq(12) &
    df["phone"].str.startswith("91", na=False),
    "phone"
].str[-10:]

# Invalid phone numbers
invalid_phone_mask = (
    df["phone"].notna() &
    ~df["phone"].str.len().eq(10)
)

print("Invalid phones:", invalid_phone_mask.sum())

# Make invalid phones missing
df.loc[invalid_phone_mask, "phone"] = pd.NA

# Duplicate phone numbers
duplicate_phone_mask = (
    df["phone"].notna() &
    df["phone"].duplicated(keep="first")
)

print("Duplicate phone records:", duplicate_phone_mask.sum())

# Remove duplicate phone values
df.loc[duplicate_phone_mask, "phone"] = pd.NA

print("Missing phones:", df["phone"].isna().sum())
print("Duplicate phones after cleaning:",
      df["phone"].dropna().duplicated().sum())

Invalid phones: 0
Duplicate phone records: 0
Missing phones: 242
Duplicate phones after cleaning: 0


In [72]:
df["phone"].head(20)

0     6539082113
1     8587291471
2     8783936422
3           <NA>
4     6155960236
5     6726085932
6     9873901434
7     6198189635
8           <NA>
9     6801004080
10    9067817618
11    6352919280
12    8080843592
13          <NA>
14          <NA>
15    6933034525
16    9781558885
17    6748827264
18    9254418199
19    7044313380
Name: phone, dtype: string

In [73]:
df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
df['date_of_joining'] = pd.to_datetime(df['date_of_joining'], errors='coerce')
df['exit_date'] = pd.to_datetime(df['exit_date'], errors='coerce')

print("Invalid DOB:", df['dob'].isna().sum())
print("Invalid Joining Date:", df['date_of_joining'].isna().sum())
print("Invalid Exit Date:", df['exit_date'].isna().sum())

Invalid DOB: 0
Invalid Joining Date: 0
Invalid Exit Date: 2532


In [75]:
df.loc[
    df['employment_status'].isin(['Active', 'On Notice Period']),
    'exit_date'
] = pd.NaT

print("Active/Notice employees with exit date:",
      df.loc[
          df['employment_status'].isin(['Active', 'On Notice Period']),
          'exit_date'
      ].notna().sum())

Active/Notice employees with exit date: 0


In [76]:
df['experience_years'] = pd.to_numeric(
    df['experience_years'],
    errors='coerce'
)

df.loc[df['experience_years'] < 0, 'experience_years'] = pd.NA

print("Missing experience:", df['experience_years'].isna().sum())

Missing experience: 135


In [77]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Duplicate rows removed:", before - after)
print("Final rows:", after)

Duplicate rows removed: 0
Final rows: 3200


In [78]:
print("Duplicate Employee IDs:",
      df['employee_id'].duplicated().sum())

Duplicate Employee IDs: 0


In [80]:
df[df["experience_years"] > df["age"]][
    ["employee_id", "age", "experience_years"]
].head(20)

,employee_id,age,experience_years


In [81]:
invalid_experience = (
    df["experience_years"].notna()
    & (df["experience_years"] > df["age"])
)

df.loc[invalid_experience, "experience_years"] = np.nan

In [82]:
median_satisfaction = df["satisfaction_score"].median()

print("Median:", median_satisfaction)

Median: 3.0


In [83]:
df["satisfaction_score"] = df["satisfaction_score"].fillna(
    median_satisfaction
)

In [84]:
df["satisfaction_score"].isnull().sum()

np.int64(0)

In [85]:
pd.crosstab(
    df["employment_status"],
    df["exit_date"].isna()
)

exit_date,False,True
employment_status,,
Active,0,1788
On Notice Period,0,485
Resigned,341,132
Terminated,327,127


In [86]:
df[
    df["employment_status"].isin(["Resigned", "Terminated"])
    & df["exit_date"].isna()
][
    ["employee_id", "employment_status", "exit_date"]
].head(20)

,employee_id,employment_status,exit_date
2,EMP00003,Terminated,NaT
9,EMP00010,Resigned,NaT
16,EMP00017,Terminated,NaT
29,EMP00030,Resigned,NaT
38,EMP00039,Resigned,NaT
39,EMP00040,Terminated,NaT
57,EMP00058,Resigned,NaT
72,EMP00073,Terminated,NaT
103,EMP00104,Terminated,NaT
118,EMP00119,Resigned,NaT


In [87]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Rows: 3200
Columns: 19

Missing Values:
employee_id                0
employee_name              0
gender                     0
dob                        0
age                        0
marital_status             0
city                       0
state                      0
phone                    242
email                     90
department_id              0
designation                0
education                  0
experience_years         135
date_of_joining            0
employment_status          0
exit_date               2532
monthly_basic_salary       0
satisfaction_score         0
dtype: int64

Duplicate Rows:
0

Data Types:
employee_id                        str
employee_name                      str
gender                             str
dob                     datetime64[us]
age                              int64
marital_status                     str
city                               str
state                              str
phone                           string
email        

In [88]:
for col in ["dob", "date_of_joining", "exit_date"]:
    df[col] = df[col].dt.strftime("%Y-%m-%d")

df.to_csv(
    "employees_cleaned.csv",
    index=False
)

print("employees_cleaned.csv successfully saved!")

employees_cleaned.csv successfully saved!


In [89]:
df = pd.read_csv("output/job_applications.csv")
df.head()

,application_id,candidate_name,gender,department_id,position_applied,recruitment_source,application_date,years_of_experience,expected_salary,status,phone,email
0,APP000001,Ishaan Chopra,Other,DEP99298,Intern,Indeed,2024-05-27,9.3,20000,Interviewed,+916598773533,ishaan.chopra831@yahoo.com
1,APP000002,Reyansh Bansal,Male,DEP007,Senior Analyst,Naukri,2024-11-24,0.5,45000,Applied,+917621279902,reyansh.bansal364@hotmail.com
2,APP000003,Saanvi Desai,Female,DEP008,Analyst,Indeed,2023-09-19,13.0,20000,Interviewed,+917460407348,saanvi.desai@rediffmail
3,APP000004,Kavita Menon,Female,DEP008,Vice President,Recruitment Agency,2026-03-11,13.4,90000,Shortlisted,+916874834937,kavita.menon581@yahoo.com
4,APP000005,Manish Patel,Male,DEP004,Manager,Company Website,2026-06-20,15.4,20000,Hired,+918197772771,manish.patel451@hotmail.com


In [91]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Rows: 26390
Columns: 12

Column Names:
['application_id', 'candidate_name', 'gender', 'department_id', 'position_applied', 'recruitment_source', 'application_date', 'years_of_experience', 'expected_salary', 'status', 'phone', 'email']

Data Types:
application_id             str
candidate_name             str
gender                     str
department_id              str
position_applied           str
recruitment_source         str
application_date           str
years_of_experience    float64
expected_salary          int64
status                     str
phone                      str
email                      str
dtype: object


In [92]:
missing = df.isnull().sum()

print("Missing values in each column:")
print(missing)

Missing values in each column:
application_id           0
candidate_name           0
gender                   0
department_id            0
position_applied         0
recruitment_source       0
application_date         0
years_of_experience      0
expected_salary          0
status                   0
phone                  405
email                  715
dtype: int64


In [93]:
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 390


In [94]:
duplicate_ids = df["application_id"].duplicated().sum()

print("Duplicate application IDs:", duplicate_ids)

Duplicate application IDs: 390


In [95]:
df = df.drop_duplicates()

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate application IDs:", df["application_id"].duplicated().sum())
print("Total rows:", len(df))

Duplicate rows: 0
Duplicate application IDs: 0
Total rows: 26000


In [96]:
name_spaces = (
    df["candidate_name"].notna() &
    (df["candidate_name"] != df["candidate_name"].str.strip())
).sum()

print("Names with extra spaces:", name_spaces)

Names with extra spaces: 218


In [97]:
df["candidate_name"] = df["candidate_name"].str.strip()

print("Extra spaces removed.")

Extra spaces removed.


In [98]:
valid_departments = [f"DEP{i:03d}" for i in range(1, 13)]

invalid_department = ~df["department_id"].isin(valid_departments)

print("Invalid department IDs:", invalid_department.sum())

Invalid department IDs: 486


In [99]:
df.loc[invalid_department, "department_id"] = np.nan

print("Invalid department IDs converted to missing values.")

Invalid department IDs converted to missing values.


In [100]:
df["application_date"] = pd.to_datetime(
    df["application_date"],
    errors="coerce"
)

print("Invalid application dates:", df["application_date"].isna().sum())

Invalid application dates: 0


In [101]:
invalid_exp = (
    (df["years_of_experience"] < 0) |
    (df["years_of_experience"] > 20)
)

print("Invalid experience values:", invalid_exp.sum())

Invalid experience values: 227


In [102]:
df.loc[invalid_exp, "years_of_experience"] = np.nan

print("Invalid experience values converted to missing.")

Invalid experience values converted to missing.


In [103]:
print(sorted(df["expected_salary"].dropna().unique()))

[np.int64(20000), np.int64(30000), np.int64(45000), np.int64(60000), np.int64(90000), np.int64(120000), np.int64(180000)]


In [104]:
valid_salary = [20000, 30000, 45000, 60000, 90000, 120000, 180000]

invalid_salary = ~df["expected_salary"].isin(valid_salary)

print("Invalid salary values:", invalid_salary.sum())

Invalid salary values: 0


In [105]:
print(df["gender"].value_counts(dropna=False))

gender
Other     8737
Female    8664
Male      8599
Name: count, dtype: int64


In [106]:
print(df["position_applied"].value_counts())

position_applied
Manager                   1925
General Manager           1905
Senior Associate          1895
Team Lead                 1886
Senior Analyst            1882
Executive                 1865
Consultant                1858
Intern                    1856
Senior Manager            1849
Analyst                   1843
Assistant Manager         1838
Deputy General Manager    1836
Associate                 1818
Vice President            1744
Name: count, dtype: int64


In [107]:
print(df["recruitment_source"].value_counts())

recruitment_source
Company Website       3010
Recruitment Agency    2955
Campus Placement      2924
Referral              2866
Walk-in               2864
Internshala           2856
Naukri                2854
LinkedIn              2854
Indeed                2817
Name: count, dtype: int64


In [108]:
print(df["status"].value_counts())

status
Applied        7794
Shortlisted    5663
Rejected       4616
Interviewed    4238
Hired          2637
On Hold        1052
Name: count, dtype: int64


In [109]:
phone = df["phone"].astype("string").str.strip()

valid_phone = phone.str.fullmatch(
    r"\+91[6-9]\d{9}",
    na=False
)

print("Valid phone numbers:", valid_phone.sum())
print("Invalid phone numbers:", (~valid_phone & phone.notna()).sum())
print("Missing phone numbers:", phone.isna().sum())

Valid phone numbers: 23991
Invalid phone numbers: 1610
Missing phone numbers: 399


In [110]:
df.loc[~valid_phone & phone.notna(), "phone"] = np.nan

print("Invalid phone numbers converted to missing.")

Invalid phone numbers converted to missing.


In [111]:
df["phone"] = df["phone"].str.strip()

In [112]:
email = df["email"].astype("string").str.strip()

valid_email = email.str.fullmatch(
    r"[^@\s]+@[^@\s]+\.[^@\s]+",
    na=False
)

print("Valid emails:", valid_email.sum())
print("Invalid emails:", (~valid_email & email.notna()).sum())
print("Missing emails:", email.isna().sum())

Valid emails: 23904
Invalid emails: 1390
Missing emails: 706


In [113]:
df.loc[~valid_email & email.notna(), "email"] = np.nan

df["email"] = df["email"].str.strip()

print("Invalid emails converted to missing.")

Invalid emails converted to missing.


In [114]:
print("FINAL DATA QUALITY CHECK")
print("=" * 50)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDuplicate application IDs:",
      df["application_id"].duplicated().sum())

print("\nInvalid dates:",
      df["application_date"].isna().sum())

print("\nInvalid experience:",
      ((df["years_of_experience"] < 0) |
       (df["years_of_experience"] > 20)).sum())

print("\nInvalid department IDs:",
      (~df["department_id"].isin(valid_departments) &
       df["department_id"].notna()).sum())

print("\nData types:")
print(df.dtypes)

FINAL DATA QUALITY CHECK
Rows: 26000
Columns: 12

Missing values:
application_id            0
candidate_name            0
gender                    0
department_id           486
position_applied          0
recruitment_source        0
application_date          0
years_of_experience     227
expected_salary           0
status                    0
phone                  2009
email                  2096
dtype: int64

Duplicate rows: 0

Duplicate application IDs: 0

Invalid dates: 0

Invalid experience: 0

Invalid department IDs: 0

Data types:
application_id                    str
candidate_name                    str
gender                            str
department_id                     str
position_applied                  str
recruitment_source                str
application_date       datetime64[us]
years_of_experience           float64
expected_salary                 int64
status                            str
phone                             str
email                             str

In [115]:
df.to_csv("job_applications_cleaned.csv", index=False)

print("Cleaned file saved successfully.")

Cleaned file saved successfully.


In [7]:
df = pd.read_csv("output/leaves.csv")
df.head()

,leave_id,employee_id,leave_type,start_date,end_date,days_taken,status
0,LV00001,EMP02166,Sick Leave,2026-04-29,2026-05-06,7,Rejected
1,LV00002,EMP02394,Maternity Leave,2025-12-17,2025-12-20,3,Pending
2,LV00003,EMP00868,Casual Leave,2026-04-20,2026-04-26,6,Rejected
3,LV00004,EMP01611,Unpaid Leave,2026-05-22,2026-05-24,2,Pending
4,LV00005,EMP00862,Maternity Leave,2025-03-24,2025-03-26,2,Rejected


In [6]:
leaves = pd.read_csv(
    r"C:\Users\kiran\Downloads\HR Analytics Dashboard\HR Analytics Dashboard\output\leaves.csv"
)

print("Rows:", leaves.shape[0])
print("Columns:", leaves.shape[1])

Rows: 5304
Columns: 7


In [7]:
print(leaves.columns.tolist())

['leave_id', 'employee_id', 'leave_type', 'start_date', 'end_date', 'days_taken', 'status']


In [8]:
print(leaves.dtypes)

leave_id         str
employee_id      str
leave_type       str
start_date       str
end_date         str
days_taken     int64
status           str
dtype: object


In [9]:
leaves["start_date"] = pd.to_datetime(
    leaves["start_date"], errors="coerce"
)

leaves["end_date"] = pd.to_datetime(
    leaves["end_date"], errors="coerce"
)

leaves["days_taken"] = pd.to_numeric(
    leaves["days_taken"], errors="coerce"
)

print(leaves.dtypes)

leave_id                  str
employee_id               str
leave_type                str
start_date     datetime64[us]
end_date       datetime64[us]
days_taken              int64
status                    str
dtype: object


In [10]:
print(leaves.isnull().sum())

leave_id       0
employee_id    0
leave_type     0
start_date     0
end_date       0
days_taken     0
status         0
dtype: int64


In [11]:
text_columns = [
    "leave_id",
    "employee_id",
    "leave_type",
    "status"
]

for col in text_columns:
    leaves[col] = leaves[col].astype(str).str.strip()

In [12]:
invalid_leave_id = ~leaves["leave_id"].str.fullmatch(r"LV\d{5}")

print("Invalid Leave IDs:", invalid_leave_id.sum())

Invalid Leave IDs: 0


In [13]:
invalid_employee_id = ~leaves["employee_id"].str.fullmatch(r"EMP\d{5}")

print("Invalid Employee IDs:", invalid_employee_id.sum())

Invalid Employee IDs: 0


In [14]:
valid_leave_types = [
    "Casual Leave",
    "Earned Leave",
    "Maternity Leave",
    "Paternity Leave",
    "Sick Leave",
    "Unpaid Leave"
]

invalid_leave_type = ~leaves["leave_type"].isin(valid_leave_types)

print("Invalid Leave Types:", invalid_leave_type.sum())

Invalid Leave Types: 0


In [15]:
valid_status = [
    "Approved",
    "Rejected",
    "Pending"
]

invalid_status = ~leaves["status"].isin(valid_status)

print("Invalid Status:", invalid_status.sum())

Invalid Status: 0


In [16]:
print("Invalid Start Dates:",
      leaves["start_date"].isna().sum())

print("Invalid End Dates:",
      leaves["end_date"].isna().sum())

Invalid Start Dates: 0
Invalid End Dates: 0


In [17]:
invalid_date_order = (
    leaves["end_date"] < leaves["start_date"]
)

print(
    "End Date Before Start Date:",
    invalid_date_order.sum()
)

End Date Before Start Date: 0


In [18]:
calculated_days = (
    leaves["end_date"] - leaves["start_date"]
).dt.days

In [19]:
days_mismatch = (
    leaves["days_taken"] != calculated_days
)

print(
    "Days Taken Mismatch:",
    days_mismatch.sum()
)

Days Taken Mismatch: 0


In [20]:
print("Minimum Days:", leaves["days_taken"].min())
print("Maximum Days:", leaves["days_taken"].max())

Minimum Days: 1
Maximum Days: 10


In [21]:
invalid_days = leaves["days_taken"] <= 0

print("Zero/Negative Days:", invalid_days.sum())

Zero/Negative Days: 0


In [22]:
print("Duplicate Rows:", leaves.duplicated().sum())

Duplicate Rows: 104


In [23]:
print(
    "Duplicate Leave IDs:",
    leaves["leave_id"].duplicated().sum()
)

Duplicate Leave IDs: 104


In [25]:
duplicate_records = leaves[
    leaves["leave_id"].duplicated(keep=False)
].sort_values("leave_id")

print(duplicate_records.head(10))

     leave_id employee_id       leave_type start_date   end_date  days_taken  \
8     LV00009    EMP01932  Paternity Leave 2026-06-30 2026-07-01           1   
5284  LV00009    EMP01932  Paternity Leave 2026-06-30 2026-07-01           1   
18    LV00019    EMP01165  Maternity Leave 2025-04-21 2025-04-24           3   
5237  LV00019    EMP01165  Maternity Leave 2025-04-21 2025-04-24           3   
24    LV00025    EMP02391  Maternity Leave 2026-05-25 2026-05-31           6   
5224  LV00025    EMP02391  Maternity Leave 2026-05-25 2026-05-31           6   
91    LV00092    EMP00454  Paternity Leave 2025-08-04 2025-08-05           1   
5258  LV00092    EMP00454  Paternity Leave 2025-08-04 2025-08-05           1   
100   LV00101    EMP02674     Earned Leave 2026-06-28 2026-07-04           6   
5296  LV00101    EMP02674     Earned Leave 2026-06-28 2026-07-04           6   

        status  
8      Pending  
5284   Pending  
18    Approved  
5237  Approved  
24    Approved  
5224  Approved  


In [26]:
leaves = leaves.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", leaves.shape[0])

Rows after removing duplicates: 5200


In [27]:
print("Duplicate Rows:",
      leaves.duplicated().sum())

print("Duplicate Leave IDs:",
      leaves["leave_id"].duplicated().sum())

Duplicate Rows: 0
Duplicate Leave IDs: 0


In [28]:
print(leaves.isnull().sum())

leave_id       0
employee_id    0
leave_type     0
start_date     0
end_date       0
days_taken     0
status         0
dtype: int64


In [29]:
print(leaves.dtypes)

leave_id                  str
employee_id               str
leave_type                str
start_date     datetime64[us]
end_date       datetime64[us]
days_taken              int64
status                    str
dtype: object


In [31]:
print("===== FINAL DATA QUALITY CHECK =====")

print("Rows:", leaves.shape[0])
print("Columns:", leaves.shape[1])

print("\nMissing Values:")
print(leaves.isnull().sum())

print("\nDuplicate Rows:",
      leaves.duplicated().sum())

print("Duplicate Leave IDs:",
      leaves["leave_id"].duplicated().sum())

print(
    "Invalid Leave IDs:",
    (~leaves["leave_id"].str.fullmatch(r"LV\d{5}")).sum()
)

print(
    "Invalid Employee IDs:",
    (~leaves["employee_id"].str.fullmatch(r"EMP\d{5}")).sum()
)

print(
    "End Date Before Start Date:",
    (leaves["end_date"] < leaves["start_date"]).sum()
)

calculated_days = (
    leaves["end_date"] - leaves["start_date"]
).dt.days

print(
    "Days Taken Mismatch:",
    (leaves["days_taken"] != calculated_days).sum()
)

print("\nFinal Data:")
print(leaves.head())

===== FINAL DATA QUALITY CHECK =====
Rows: 5200
Columns: 7

Missing Values:
leave_id       0
employee_id    0
leave_type     0
start_date     0
end_date       0
days_taken     0
status         0
dtype: int64

Duplicate Rows: 0
Duplicate Leave IDs: 0
Invalid Leave IDs: 0
Invalid Employee IDs: 0
End Date Before Start Date: 0
Days Taken Mismatch: 0

Final Data:
  leave_id employee_id       leave_type start_date   end_date  days_taken  \
0  LV00001    EMP02166       Sick Leave 2026-04-29 2026-05-06           7   
1  LV00002    EMP02394  Maternity Leave 2025-12-17 2025-12-20           3   
2  LV00003    EMP00868     Casual Leave 2026-04-20 2026-04-26           6   
3  LV00004    EMP01611     Unpaid Leave 2026-05-22 2026-05-24           2   
4  LV00005    EMP00862  Maternity Leave 2025-03-24 2025-03-26           2   

     status  
0  Rejected  
1   Pending  
2  Rejected  
3   Pending  
4  Rejected  


In [32]:
leaves.to_csv(
    "leaves_cleaned.csv",
    index=False
)

print("leaves_cleaned.csv saved successfully!")

leaves_cleaned.csv saved successfully!


In [35]:
payroll = pd.read_csv(
    r"C:\Users\kiran\Downloads\HR Analytics Dashboard\HR Analytics Dashboard\output\payroll.csv"
)

print("Rows:", payroll.shape[0])
print("Columns:", payroll.shape[1])

Rows: 32480
Columns: 9


In [36]:
print(payroll.columns.tolist())

['payroll_id', 'employee_id', 'pay_month', 'basic_salary', 'hra', 'bonus', 'deductions', 'net_pay', 'payment_mode']


In [37]:
df.head()

,payroll_id,employee_id,pay_month,basic_salary,hra,bonus,deductions,net_pay,payment_mode
0,PAY000001,EMP97793,2025-11,25000,10000.0,2000,1902.57,35097.43,Cash
1,PAY000002,EMP01893,2025-09,25000,10000.0,5000,2319.19,37680.81,Cheque
2,PAY000003,EMP00146,2025-04,160000,64000.0,0,15178.69,208821.31,Cash
3,PAY000004,EMP01221,2025-05,160000,64000.0,5000,19467.13,209532.87,Cheque
4,PAY000005,EMP00147,2026-03,160000,64000.0,0,21447.81,202552.19,Bank Transfer


In [38]:
df.tail()

,payroll_id,employee_id,pay_month,basic_salary,hra,bonus,deductions,net_pay,payment_mode
32475,PAY014399,EMP02964,2026-05,320000,128000.0,25000,39898.84,433101.16,Cash
32476,PAY008851,EMP02911,2026-05,250000,100000.0,5000,34472.80,320527.20,Bank Transfer
32477,PAY022180,EMP00007,2025-04,25000,10000.0,0,1989.69,33010.31,Cheque
32478,PAY011127,EMP02008,2026-04,160000,64000.0,10000,10520.33,223479.67,Cheque
32479,PAY022884,EMP03182,2025-05,25000,10000.0,15000,1752.37,48247.63,Cash


In [39]:
df.dtypes

payroll_id          str
employee_id         str
pay_month           str
basic_salary      int64
hra             float64
bonus             int64
deductions      float64
net_pay         float64
payment_mode        str
dtype: object

In [40]:
df["pay_month"] = pd.to_datetime(df["pay_month"], format="%Y-%m", errors="coerce")

print("Invalid pay_month:", df["pay_month"].isna().sum())

Invalid pay_month: 0


In [41]:
print(df.isnull().sum())

payroll_id        0
employee_id       0
pay_month         0
basic_salary      0
hra               0
bonus             0
deductions        0
net_pay         634
payment_mode      0
dtype: int64


In [42]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 480


In [43]:
df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing exact duplicates:", len(df))

Rows after removing exact duplicates: 32000


In [44]:
print("Duplicate payroll IDs:", df["payroll_id"].duplicated().sum())

Duplicate payroll IDs: 0


In [45]:
invalid_payroll_id = ~df["payroll_id"].astype(str).str.fullmatch(r"PAY\d{6}")

print("Invalid payroll IDs:", invalid_payroll_id.sum())

Invalid payroll IDs: 0


In [46]:
invalid_employee_id = ~df["employee_id"].astype(str).str.fullmatch(r"EMP\d{5}")

print("Invalid employee IDs:", invalid_employee_id.sum())

Invalid employee IDs: 0


In [47]:
df["pay_month"] = pd.to_datetime(
    df["pay_month"],
    format="%Y-%m",
    errors="coerce"
)

In [48]:
print("Invalid pay months:", df["pay_month"].isna().sum())

Invalid pay months: 0


In [49]:
numeric_columns = [
    "basic_salary",
    "hra",
    "bonus",
    "deductions",
    "net_pay"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_columns].dtypes)

basic_salary      int64
hra             float64
bonus             int64
deductions      float64
net_pay         float64
dtype: object


In [50]:
for col in numeric_columns:
    print(col, "negative values:", (df[col] < 0).sum())

basic_salary negative values: 0
hra negative values: 0
bonus negative values: 0
deductions negative values: 0
net_pay negative values: 0


In [51]:
print(df["payment_mode"].unique())

<StringArray>
['Cash', 'Cheque', 'Bank Transfer']
Length: 3, dtype: str


In [52]:
valid_payment_modes = ["Cash", "Cheque", "Bank Transfer"]

invalid_payment_mode = ~df["payment_mode"].isin(valid_payment_modes)

print("Invalid payment modes:", invalid_payment_mode.sum())

Invalid payment modes: 0


In [55]:
expected_hra = df["basic_salary"] * 0.40

invalid_hra = ~np.isclose(
    df["hra"],
    expected_hra,
    atol=0.01
)

print("Incorrect HRA:", invalid_hra.sum())

Incorrect HRA: 0


In [56]:
expected_net_pay = (
    df["basic_salary"]
    + df["hra"]
    + df["bonus"]
    - df["deductions"]
)

wrong_net_pay = ~np.isclose(
    df["net_pay"],
    expected_net_pay,
    atol=0.01,
    equal_nan=True
)

print("Incorrect net pay:", wrong_net_pay.sum())

Incorrect net pay: 931


In [57]:
df["net_pay"]= expected_net_pay

In [58]:
expected_net_pay = (
    df["basic_salary"]
    + df["hra"]
    + df["bonus"]
    - df["deductions"]
)

check = np.isclose(
    df["net_pay"],
    expected_net_pay,
    atol=0.01
)

print("Incorrect net pay after cleaning:", (~check).sum())

Incorrect net pay after cleaning: 0


In [59]:
duplicate_employee_month = df.duplicated(
    subset=["employee_id", "pay_month"]
).sum()

print("Employee + Pay Month duplicates:", duplicate_employee_month)

Employee + Pay Month duplicates: 6958


In [60]:
print(df.isnull().sum())

payroll_id      0
employee_id     0
pay_month       0
basic_salary    0
hra             0
bonus           0
deductions      0
net_pay         0
payment_mode    0
dtype: int64


In [61]:
print("Final duplicate rows:", df.duplicated().sum())
print("Final duplicate payroll IDs:", df["payroll_id"].duplicated().sum())

Final duplicate rows: 0
Final duplicate payroll IDs: 0


In [62]:
print(df.shape)
print(df.info())

(32000, 9)
<class 'pandas.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   payroll_id    32000 non-null  str           
 1   employee_id   32000 non-null  str           
 2   pay_month     32000 non-null  datetime64[us]
 3   basic_salary  32000 non-null  int64         
 4   hra           32000 non-null  float64       
 5   bonus         32000 non-null  int64         
 6   deductions    32000 non-null  float64       
 7   net_pay       32000 non-null  float64       
 8   payment_mode  32000 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(2), str(3)
memory usage: 2.2 MB
None


In [63]:
df["pay_month"] = df["pay_month"].dt.strftime("%Y-%m")

In [64]:
df.to_csv("payroll_cleaned.csv", index=False)

print("payroll_cleaned.csv created successfully!")

payroll_cleaned.csv created successfully!


In [4]:
performance_reviews = pd.read_csv(
    r"C:\Users\kiran\Downloads\HR Analytics Dashboard\HR Analytics Dashboard\output\performance_reviews.csv"
)

print("Rows:", performance_reviews.shape[0])
print("Columns:", performance_reviews.shape[1])

Rows: 6293
Columns: 7


In [6]:
performance_reviews.columns

Index(['review_id', 'employee_id', 'review_period', 'performance_rating',
       'productivity_score', 'promotion_recommended', 'manager_comments_flag'],
      dtype='str')

In [7]:
performance_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 6293 entries, 0 to 6292
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   review_id              6293 non-null   str    
 1   employee_id            6293 non-null   str    
 2   review_period          6293 non-null   str    
 3   performance_rating     6092 non-null   float64
 4   productivity_score     6293 non-null   float64
 5   promotion_recommended  6293 non-null   str    
 6   manager_comments_flag  6293 non-null   str    
dtypes: float64(2), str(5)
memory usage: 344.3 KB


In [8]:
missing_values = performance_reviews.isnull().sum()

print(missing_values)

review_id                  0
employee_id                0
review_period              0
performance_rating       201
productivity_score         0
promotion_recommended      0
manager_comments_flag      0
dtype: int64


In [10]:
print("Duplicate rows:", performance_reviews.duplicated().sum())

Duplicate rows: 93


In [11]:
performance_reviews = performance_reviews.drop_duplicates()

print("Rows after removing duplicates:", len(performance_reviews))

Rows after removing duplicates: 6200


In [13]:
text_columns = [
    "review_id",
    "employee_id",
    "review_period",
    "promotion_recommended",
    "manager_comments_flag"
]

for col in text_columns:
   performance_reviews [col] = performance_reviews[col].astype("string").str.strip()

In [14]:
performance_reviews["promotion_recommended"] = (
    performance_reviews["promotion_recommended"]
    .str.strip()
    .str.title()
)

performance_reviews["manager_comments_flag"] = (
    performance_reviews["manager_comments_flag"]
    .str.strip()
    .str.title()
)

In [15]:
performance_reviews["review_period"] = (
    performance_reviews["review_period"]
    .str.strip()
    .str.upper()
)

In [16]:
print(performance_reviews["review_period"].unique())

<StringArray>
['2025-H2', '2026-H1', '2024-H1', '2025-H1', '2024-H2']
Length: 5, dtype: string


In [17]:
print(performance_reviews["performance_rating"].isnull().sum())

print(performance_reviews["performance_rating"].value_counts(dropna=False).sort_index())

197
performance_rating
1.0     315
2.0     557
3.0    2147
4.0    2043
5.0     941
NaN     197
Name: count, dtype: int64


In [18]:
performance_reviews["performance_rating"] = performance_reviews["performance_rating"].fillna(3)

In [19]:
performance_reviews["performance_rating"] = performance_reviews["performance_rating"].astype(int)

In [20]:
print("Minimum:", performance_reviews["productivity_score"].min())
print("Maximum:", performance_reviews["productivity_score"].max())
print("Missing:", performance_reviews["productivity_score"].isnull().sum())

Minimum: 0.0
Maximum: 99.9
Missing: 0


In [21]:
performance_reviews["productivity_score"] = pd.to_numeric(
    performance_reviews["productivity_score"],
    errors="coerce"
)

In [22]:
print(performance_reviews["productivity_score"].isnull().sum())

0


In [23]:
invalid_review_ids = performance_reviews[
    ~performance_reviews["review_id"].str.match(r"^PRF\d{5}$", na=False)
]

print("Invalid review IDs:", len(invalid_review_ids))

Invalid review IDs: 0


In [24]:
invalid_employee_ids = performance_reviews[
    ~performance_reviews["employee_id"].str.match(r"^EMP\d{5}$", na=False)
]

print("Invalid employee IDs:", len(invalid_employee_ids))

Invalid employee IDs: 0


In [25]:
valid_periods = [
    "2024-H1",
    "2024-H2",
    "2025-H1",
    "2025-H2",
    "2026-H1"
]

invalid_periods = performance_reviews[
    ~performance_reviews["review_period"].isin(valid_periods)
]

print("Invalid review periods:", len(invalid_periods))

Invalid review periods: 0


In [26]:
print("Promotion values:")
print(performance_reviews["promotion_recommended"].value_counts())

print("\nManager comments values:")
print(performance_reviews["manager_comments_flag"].value_counts())

Promotion values:
promotion_recommended
No     4941
Yes    1259
Name: count, dtype: Int64

Manager comments values:
manager_comments_flag
No     3122
Yes    3078
Name: count, dtype: Int64


In [27]:
print(performance_reviews.isnull().sum())

review_id                0
employee_id              0
review_period            0
performance_rating       0
productivity_score       0
promotion_recommended    0
manager_comments_flag    0
dtype: int64


In [28]:
print("Duplicate rows:",performance_reviews.duplicated().sum())

Duplicate rows: 0


In [29]:
print(performance_reviews.shape)
print(performance_reviews.info())
print(performance_reviews.head(10))

(6200, 7)
<class 'pandas.DataFrame'>
RangeIndex: 6200 entries, 0 to 6199
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   review_id              6200 non-null   string 
 1   employee_id            6200 non-null   string 
 2   review_period          6200 non-null   string 
 3   performance_rating     6200 non-null   int64  
 4   productivity_score     6200 non-null   float64
 5   promotion_recommended  6200 non-null   string 
 6   manager_comments_flag  6200 non-null   string 
dtypes: float64(1), int64(1), string(5)
memory usage: 339.2 KB
None
  review_id employee_id review_period  performance_rating  productivity_score  \
0  PRF00001    EMP01873       2025-H2                   2                69.9   
1  PRF00002    EMP02127       2026-H1                   3                80.9   
2  PRF00003    EMP02652       2026-H1                   5                41.0   
3  PRF00004    EMP00110       2025-

In [30]:
performance_reviews.to_csv("performance_reviews_cleaned.csv", index=False)

print("Cleaned file saved successfully!")

Cleaned file saved successfully!
